# Introduction
Welcome to Lab 2, in this lab we will take our first steps into applying ML to satellite data, using landcover analysis as our primary goal.

As covered in Lecture 2, landcover classification is the quintessential remote sensing task in the civil sector. The LandSat programme (Lecture 3) was originally developed in large part to enable the USA to monitor landcover more easily for a range of agricultral, govermental and finacial problemsets. It remains a primary task for governments and companies around the world.

This is a pixel-wise ML task. We will apply two methods, both of which are available in GEE as ready-to-go algorithms. In this instance we are happy to go with built-in as it allows us to become familiar with an end-to-end GEE ML workflow without getting bogged down in technical detail. No need to re-invent the wheel unless you are going to the [moon](https://www.nasa.gov/feature/ames/artemis-moon-rover-s-wheels-are-ready-to-roll/).

Remember that this is a set of exercises and tutorials that escalate in difficulty. Always show your working and do not neccesarily expect to get full marks on everything.

Use the resources available to you outside of this material. I expect you to search for concepts or terms you do not understand and to look at the documentation of the code packages that we are using in the first instance of getting stuck. But do then of course ask us for help in the lab and on EdDiscussion.
_______________________________________________________________________________

# Set up

In [ ]:
# Install libs if needed, note the --quiet option means a lot less annoying info than as Lab 1 when we did the same!
!pip install geemap --quiet

In [ ]:
# Set up GEE API
import ee
ee.Authenticate()
ee.Initialize(project='geog761-ai') #<- Remember to change this to your own project's name!

In [ ]:
# Import other libs
import geemap
import geopandas as gpd
import pandas as pd

# Define Area of Interest (AOI)

In [ ]:
# Define area of interest (e.g., Wellington, NZ)
aoi = ee.Geometry.Rectangle([174.6, -41.4, 174.9, -41.2])
Map = geemap.Map(center=[-41.3, 174.75], zoom=10)
Map.addLayer(aoi, {}, 'AOI')

# Visualize on the map to check got it right... (always a good idea)
Map

# Load and Cloud Mask Imagery from Earth Engine

In [ ]:
# Set up your filter Sentinel-2 imagery
# A function that masks clouds in your S2 images via QA band
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000).select(['B2', 'B3', 'B4', 'B8'])

# Now build the clean collection with selected bands
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2021-01-01', '2021-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
      .map(mask_s2_clouds)
      .median() #<- think about this step and what it is doing to your image when you build your single composite from the image collection
      .clip(aoi))

(1) What other mathematical [compositing](https://developers.google.com/earth-engine/guides/ic_composite_mosaic) approaches could you take when building your single image from the filtered image collection you create in the cell above? Hint: you currently are doing a 'median-composite'. (3 pts)

**Answer (1).** `.median()` reduces the collection band by band, taking the 50th percentile of the valid observations at each pixel. Any other reducer can be substituted at that step, and Earth Engine also offers a family of composites that *select a real observation* instead of computing a new value.*Statistical reducers (compute a new value per band)*- `.mean()` - arithmetic average of every valid observation. Uses all the data, but one missed cloud or shadow drags the value.- `.min()` / `.max()` - the classic dark-object composite (`.min()` on blue, because cloud is bright) and its opposite for bright targets such as snow. Least robust of all, because the result *is* the outlier.- `ee.Reducer.percentile([p])` - the general case of the median (p = 50). A low percentile (15-30) biases towards darker, less cloud-contaminated pixels; a high percentile towards brighter/greener ones.- `.mode()` - most frequent value; only meaningful for categorical images, e.g. compositing per-date classifications.- `ee.Reducer.median().combine(ee.Reducer.stdDev(), sharedInputs=True)` - several statistics in a single pass, e.g. to add a within-year variability band as an extra classifier feature.*Selection composites (keep a real observation)*- `.mosaic()` / `.sort('system:time_start').mosaic()` - last unmasked pixel in the stack, i.e. most-recent-pixel. Spectrally self-consistent per pixel, but visible date seams between scenes.- `.qualityMosaic('NDVI')` - keeps, per pixel, the observation maximising a chosen band: greenest-pixel composite, or least-cloudy-pixel if run on an inverted cloud-probability band.- **Medoid** - keeps the observation whose *whole* spectral vector is closest to the median vector.Why it matters here: a band-wise median reduces each band independently, so the composite spectrum at a pixel is generally one that no single acquisition ever recorded, and band ratios computed from it are slightly synthetic. A medoid or quality mosaic preserves a physically real spectrum - worth having when the composite is about to be fed to a classifier that learns from relationships between bands.

In [ ]:
# Set up the visualisation and then once again check it has worked, stacking layers in a sensible order
Map = geemap.Map(center=[-41.3, 174.75], zoom=11) #<- reset the map object to clear out other layers from earlier cells
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 RGB')
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map

________________________________________________________________________________

# Training data

We are going to train our data using the European Space Agency (ESA) world landcover map. Given that this is derived from satellite data our logic is at risk of becoming a little circular here. But as a training exercise it is a good way to have easy to access training data of the right size and type to apply to our image.

There are a wide range of landcover datasets in the GEE catalogue that you can use for this purpose:
*   https://developers.google.com/earth-engine/datasets/tags/landuse-landcover

First, we will load in the landcover map we will use to create our training dataset (https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v100):



Next we need to use our landcover map to [sample](https://developers.google.com/earth-engine/apidocs/ee-image-sample) the Sentinel 2 image, associating each pixel of a given landcover with the spectral band values 'beneath' it.

In [ ]:
# Define valid WorldCover classes (10 to 100, spaced by 10)
valid_classes = ee.List.sequence(10, 100, 10)

# Need to change this to sequential list to avoid the algs thiniing there are 99 classes rather than 9
remap_to = ee.List.sequence(1, 10)

# Load landcover map
landcover = ee.Image('ESA/WorldCover/v100/2020').select('Map').clip(aoi)

# Remap labels: create a new band with remapped values (10 → 1, ..., 100 → 10)
landcover_remapped = landcover.remap(valid_classes, remap_to).rename('Map_remapped')

(2) What year of data was this landcover map (accessed in the cell above) created on? (1 pt)

**Answer (2).** **2020.** The asset is `ESA/WorldCover/v100/2020` and its `system:time_start` is 2020-01-01, so the map represents the 2020 calendar year; ESA built it from Sentinel-1 and Sentinel-2 data acquired during 2020, and 2020 is the only year in the v100 collection.

In [ ]:
# Add landcover original and remapped as bands to Sentinel-2 image
training_data = s2.addBands(landcover_remapped).addBands(landcover_remapped)

Let's inspect the resulting training dataset and make sure that you understand what you have built here.

We are going to:
- Print the first few rows (client-side).
- Check the number of samples per class (server-side with reduceColumns).

This may take a little time to run as we request data both client and server side.

In [ ]:
# Sample the image
bands = ['B2', 'B3', 'B4', 'B8']
sample = training_data.select(bands + ['Map_remapped']).sample(
    region=aoi,
    scale=10,
    numPixels=5000,
    seed=2,
    geometries=True)

# Print first 10 sample points (client-side)
first_10 = sample.limit(10).getInfo()

print('\nFirst 10 training samples:')
for i, feature in enumerate(first_10['features']):
    props = feature['properties']
    print(f"Sample {i+1}: Class={props['Map_remapped']}, B2={props['B2']}, B3={props['B3']}, B4={props['B4']}, B8={props['B8']}")

# Count number of samples per class (server-side)
class_counts = sample.reduceColumns(
    reducer=ee.Reducer.frequencyHistogram(),
    selectors=['Map_remapped'])

print('\nClass distribution in sample:')
print(class_counts.getInfo())


(3) What type of landcover is being sampled in sample number 2 of the dataset? Make sure your random seed is set to 2 in the sampling code so that you get the same answer as me. (1 pt)

**Answer (3).** **Permanent water bodies.** With `seed = 2`, sample 2 returns `Map_remapped = 8`, and 8 is the remap of the original ESA WorldCover code 80 = permanent water bodies. The spectral values confirm it: B2 = 0.0394, B3 = 0.0288, B4 = 0.0077, B8 = 0.0052 - relatively bright in blue/green, effectively zero in red and NIR.

We are now ready to start machine learning on classifying the landcover of Wellington!
________________________________________________________________________________

# Random Forest Classification for Landcover
Let us do machine learning properly and first apply a 70/20/10 split that allows us to produce some accuracy statistics.

In [ ]:
# Add random column
sample = sample.randomColumn('random')

# Split
train = sample.filter(ee.Filter.lt('random', 0.7))
valid = sample.filter(ee.Filter.And(ee.Filter.gte('random', 0.7), ee.Filter.lt('random', 0.9)))
test = sample.filter(ee.Filter.gte('random', 0.9))

Our next task is to set up a classifier object, followed by applying it to the image using the .classify operator. This operator hides a LOT of behind the scenes plumbing from Google.

We use the [GEE Random Forest](https://developers.google.com/earth-engine/apidocs/ee-classifier-smilerandomforest) model that they have pre-built for us. Specifying the number of trees to use is the only variable we must set. Othe variables are available, read the docs!

In [ ]:
# Train the RF model
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=train,
    classProperty='Map_remapped',
    inputProperties=bands)

With the model trained, we just apply it simply using the '.classify' operator.

In [ ]:
# Classify image
classified = s2.select(bands).classify(classifier)

Let's visualize what we have just made!

In [ ]:
# Classified image viz, fresh set up again to make sure we have exactly what we want displayed.
Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(classified.randomVisualizer(), {}, 'Classified')
Map.addLayer(landcover_remapped.randomVisualizer(), {} ,'ESA Landcover')
Map

(4) Look at the ESA landcover map and our resulting random forest classified Sentinel 2 image map. How are they different and what might be some causes of this difference? (5 pts)

**Answer (4).** 89.4 % of the valid AOI pixels get the same class in both maps, and the broad structure of the scene is reproduced. The differences are of three kinds.1. **Spatial texture.** ESA WorldCover is smooth and patch-like; our map is speckled. The random forest is strictly pixel-wise - four reflectance values per pixel and no spatial context - so neighbouring pixels are labelled independently. ESA apply spatial post-processing and an effective minimum mapping unit.2. **The rare classes disappear.** The 5 000-pixel sample is heavily imbalanced (water 2614, tree 1262, grass 850, built 221, bare 36, wetland 10, shrub 6, crop 1). Shrubland falls from 0.13 % of the AOI to 0.00 %, cropland 0.04 % to 0.00 %, bare 0.72 % to 0.14 %, herbaceous wetland 0.17 % to 0.01 %. A majority vote over 100 trees will not pick a class only a few leaves ever see.3. **Systematic confusion at particular boundaries.** Tree/grass is the main pair (23 grass predicted tree, 16 tree predicted grass on the test split) and built-up is over-predicted in vegetated suburbs (precision 0.63 vs recall 0.75). Both are mixed-pixel problems.**Causes:** only four bands (no SWIR, which is what separates bare/built/dry grass; no red edge; no texture; no time series); ESA was built from a full year of Sentinel-1 + Sentinel-2 time series with a large quality-controlled training set, we have one annual median composite; our labels *are* ESA, whose reported global overall accuracy is ~74 %, so we are partly learning ESA's errors (the logic is circular, but the circle does not close); the band-wise median blends dates and Wellington's steep terrain puts slopes in permanent shadow, which pushes them towards water/built-up; and ESA generalise to a mapping unit while we do not generalise at all.

Finally, we will apply our test data split and produce the [summary statistics](https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall) that we need to make smart choices about our training approach.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd


# Validate
validated = valid.classify(classifier, 'predicted')

# Function to export data for confusion matrix
def fc_to_lists(fc, classProp, predProp):
    values = fc.aggregate_array(classProp).getInfo()
    preds = fc.aggregate_array(predProp).getInfo()
    return values, preds

# Get predicted vs actual from validation set
y_true, y_pred = fc_to_lists(validated, 'Map_remapped', 'predicted')

# Labels for original classes
label_map = {i + 1: valid_classes.get(i).getInfo() for i in range(10)}
label_names = [label_map[i + 1] for i in range(10)]

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(1, 11)))
report = classification_report(y_true, y_pred, labels=list(range(1, 11)), target_names=[str(l) for l in label_names])

# Pretty-print
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=[f"Actual {l}" for l in label_names],
                       columns=[f"Pred {l}" for l in label_names]))
print("\nClassification Report:")
print(report)

(5) Exercise: modify the code so that you feed into your random forest classifier that has been trained on 2020 data, on other years of Sentinel 2 data than 2020. Produce a publication quality figure that presents the following:
*   Landcover maps of Wellington for 2018, 2022 and 2024
*   The Test accuracy averages of the RF classifier for 2020.
*   Include a sentence in your figure caption that explains why you cannot state the accuracy of the classifier for years other than 2020.

(15 pts)
_______________________________________________________________________________


### Answer (5)One classifier is trained once and then reused unchanged. Note that the notebook as supplied trains on the **2021** composite while the labels come from ESA WorldCover **2020**; below the training composite is moved to 2020 so that imagery and labels refer to the same year, which is what "trained on 2020 data" should mean and which makes the reported accuracy interpretable. The same fitted model is then applied to the 2018, 2022 and 2024 annual median composites with no retraining.

In [ ]:
# ---- Q5: one RF trained on 2020, applied to 2018 / 2022 / 2024 ----------------import requests, io, numpy as np, matplotlib.pyplot as pltfrom matplotlib.colors import ListedColormap, BoundaryNormfrom matplotlib.patches import Rectanglefrom sklearn.metrics import (accuracy_score, cohen_kappa_score,                             precision_recall_fscore_support)import rasterio                      # !pip install rasterio if neededBANDS = ['B2', 'B3', 'B4', 'B8']def composite(year):    """Annual median composite of the low-cloud Sentinel-2 scenes over the AOI."""    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')            .filterBounds(aoi)            .filterDate(f'{year}-01-01', f'{year}-12-31')            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))            .map(mask_s2_clouds).median().clip(aoi))def n_scenes(year):    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi)            .filterDate(f'{year}-01-01', f'{year}-12-31')            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)).size().getInfo())# train on 2020 imagery + ESA WorldCover 2020 labelss2_2020 = composite(2020)sample20 = (s2_2020.addBands(landcover_remapped)            .select(BANDS + ['Map_remapped'])            .sample(region=aoi, scale=10, numPixels=5000, seed=2, geometries=True)            .randomColumn('random', 2))train20 = sample20.filter(ee.Filter.lt('random', 0.7))valid20 = sample20.filter(ee.Filter.And(ee.Filter.gte('random', 0.7),                                        ee.Filter.lt('random', 0.9)))test20  = sample20.filter(ee.Filter.gte('random', 0.9))rf = ee.Classifier.smileRandomForest(numberOfTrees=100).train(    features=train20, classProperty='Map_remapped', inputProperties=BANDS)# score the held-out test split oncetested = test20.classify(rf, 'predicted')y_true = tested.aggregate_array('Map_remapped').getInfo()y_pred = tested.aggregate_array('predicted').getInfo()oa  = accuracy_score(y_true, y_pred)kap = cohen_kappa_score(y_true, y_pred)mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, average='macro',                                                zero_division=0)wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted',                                                zero_division=0)print(f'2020 test split (n={len(y_true)}): OA={oa:.3f}  kappa={kap:.3f}')print(f'  macro    P/R/F1 = {mp:.2f} / {mr:.2f} / {mf:.2f}')print(f'  weighted P/R/F1 = {wp:.2f} / {wr:.2f} / {wf:.2f}')

In [ ]:
# ---- pull the classified rasters down so we can draw a proper figure ----------def fetch(img, region, scale=20, tiles=3):    """getDownloadURL is capped, so grab the region as a grid of tiles and mosaic.    Returns (array, extent) with array shaped (bands, rows, cols) and    extent = (left, right, bottom, top) ready for imshow.    """    from rasterio.merge import merge    c = region.bounds().coordinates().getInfo()[0]    xs = [p[0] for p in c]; ys = [p[1] for p in c]    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)    srcs = []    for i in range(tiles):        for j in range(tiles):            box = ee.Geometry.Rectangle(                [x0 + (x1-x0)*i/tiles, y0 + (y1-y0)*j/tiles,                 x0 + (x1-x0)*(i+1)/tiles, y0 + (y1-y0)*(j+1)/tiles], None, False)            url = img.getDownloadURL({'scale': scale, 'region': box,                                      'format': 'GEO_TIFF', 'crs': 'EPSG:4326'})            srcs.append(rasterio.open(io.BytesIO(requests.get(url, timeout=900).content)))    arr, _ = merge(srcs)    ext = (min(s.bounds.left for s in srcs), max(s.bounds.right for s in srcs),           min(s.bounds.bottom for s in srcs), max(s.bounds.top for s in srcs))    return arr, extYEARS = [2018, 2022, 2024]SCENES = {y: n_scenes(y) for y in YEARS + [2020]}print('cloud-filtered scenes per year:', SCENES)maps = {}for y in YEARS:    a, e = fetch(composite(y).select(BANDS).classify(rf).unmask(0).toByte(), aoi)    maps[y] = (a[0], e)

In [ ]:
# ---- Figure 2 -----------------------------------------------------------------ESA_NAMES = {1: 'Tree cover', 2: 'Shrubland', 3: 'Grassland', 4: 'Cropland',             5: 'Built-up', 6: 'Bare / sparse vegetation', 7: 'Snow and ice',             8: 'Permanent water bodies', 9: 'Herbaceous wetland', 10: 'Moss and lichen'}ESA_COLS = {0: '#ffffff', 1: '#006400', 2: '#ffbb22', 3: '#ffff4c', 4: '#f096ff',            5: '#fa0000', 6: '#b4b4b4', 7: '#f0f0f0', 8: '#0064c8', 9: '#0096a0',            10: '#fae6a0'}CMAP = ListedColormap([ESA_COLS[i] for i in range(11)])NORM = BoundaryNorm(np.arange(-0.5, 11.5, 1), 11)def scalebar(ax, ext, km=10, y=0.085, x=0.05, fs=6.0):    lat = (ext[2] + ext[3]) / 2    deg = km / (111.320 * np.cos(np.radians(lat)))    x0 = ext[0] + (ext[1]-ext[0])*x; y0 = ext[2] + (ext[3]-ext[2])*y    h = (ext[3]-ext[2])*0.012    ax.add_patch(Rectangle((x0-deg*0.06, y0-h*1.6), deg*1.12, h*4.4, fc='white',                           ec='0.35', lw=0.5, alpha=0.85, zorder=5))    ax.add_patch(Rectangle((x0, y0), deg/2, h, fc='k', zorder=6))    ax.add_patch(Rectangle((x0+deg/2, y0), deg/2, h, fc='white', ec='k', lw=0.5, zorder=6))    for f, lab in [(0, '0'), (0.5, str(km//2)), (1, f'{km} km')]:        ax.text(x0+deg*f, y0+h*1.35, lab, ha='center', va='bottom', fontsize=fs, zorder=6)fig = plt.figure(figsize=(6.268, 4.62), dpi=300)gs = fig.add_gridspec(2, 2, hspace=0.115, wspace=0.035,                      left=0.012, right=0.988, top=0.915, bottom=0.012)for k, y in enumerate(YEARS):    ax = fig.add_subplot([gs[0, 0], gs[0, 1], gs[1, 0]][k])    arr, ext = maps[y]    ax.imshow(np.ma.masked_equal(arr, 0), extent=ext, cmap=CMAP, norm=NORM,              interpolation='nearest')    ax.set_xticks([]); ax.set_yticks([])    ax.set_title(f'({chr(97+k)})  {y}', fontsize=8.5, fontweight='bold', pad=2.5, loc='left')    ax.text(0.982, 0.028, f'{SCENES[y]} cloud-filtered S2 scenes', transform=ax.transAxes,            fontsize=6.0, va='bottom', ha='right',            bbox=dict(fc='white', ec='0.6', lw=0.4, alpha=0.9, pad=1.6))    if k == 0:        scalebar(ax, ext)        ax.annotate('', xy=(0.94, 0.92), xytext=(0.94, 0.78), xycoords='axes fraction',                    arrowprops=dict(arrowstyle='-|>', lw=1.2, color='k', mutation_scale=11))        ax.text(0.94, 0.93, 'N', transform=ax.transAxes, ha='center', va='bottom',                fontsize=8, fontweight='bold')axl = fig.add_subplot(gs[1, 1]); axl.axis('off')axl.set_title('(d)  Legend and 2020 test-set accuracy', fontsize=8.5, fontweight='bold',              pad=2.5, loc='left')for i, c in enumerate([1, 2, 3, 4, 5, 6, 8, 9]):    col, row = i // 4, i % 4    x = 0.010 + col*0.505; yy = 0.975 - row*0.088    axl.add_patch(Rectangle((x, yy-0.045), 0.042, 0.052, transform=axl.transAxes,                            fc=ESA_COLS[c], ec='0.35', lw=0.45, clip_on=False))    axl.text(x+0.054, yy-0.019, f'{c} {ESA_NAMES[c]}', transform=axl.transAxes,             fontsize=6.1, va='center')axl.text(0.010, 0.585, f'Random forest, held-out 10 % test split, 2020 (n = {len(y_true)})',         transform=axl.transAxes, fontsize=6.3, fontweight='bold', va='bottom')tb = axl.table(cellText=[['Overall accuracy', f'{oa:.3f}'],                         ['Cohen kappa', f'{kap:.3f}'],                         ['Macro avg  P / R / F1', f'{mp:.2f} / {mr:.2f} / {mf:.2f}'],                         ['Weighted avg  P / R / F1', f'{wp:.2f} / {wr:.2f} / {wf:.2f}']],               colWidths=[0.56, 0.44], cellLoc='left', bbox=[0.010, 0.300, 0.980, 0.270])tb.auto_set_font_size(False); tb.set_fontsize(6.1)for (r, c), cell in tb.get_celld().items():    cell.set_linewidth(0.35); cell.set_edgecolor('0.75')    if c == 0: cell.set_facecolor('#f2f2f2')axl.text(0.010, 0.0,         'Sentinel-2 SR Harmonized (B2, B3, B4, B8), annual median composite, scenes\n'         'with cloud < 10 %, QA60 cloud/cirrus mask, 20 m. Classifier: GEE\n'         'smileRandomForest, 100 trees, trained once on the 2020 composite with ESA\n'         'WorldCover v100 (2020) labels; 5 000 pixels, seed = 2, 70/20/10 split.\n'         'Map data (c) ESA WorldCover / Copernicus Sentinel-2, in Google Earth Engine.',         transform=axl.transAxes, fontsize=5.0, color='0.3', va='bottom')fig.suptitle('Land cover of Wellington in 2018, 2022 and 2024 from a random forest '             'trained only on 2020 data', fontsize=7.9, fontweight='bold',             x=0.012, ha='left', y=0.984)fig.savefig('fig_q5.png', dpi=300, facecolor='white')plt.show()

**Figure 2.** Land cover of Wellington in 2018, 2022 and 2024, produced by applying one random forest - trained once on the 2020 Sentinel-2 composite with ESA WorldCover v100 (2020) labels - to the annual median composite of each year. Panel (d) gives the legend and the accuracy of that classifier on the 10 % of the 2020 sample held out of training.**Why the accuracy cannot be stated for the other years:** the accuracy figures were measured against ESA WorldCover, which exists for 2020 only, so for 2018, 2022 and 2024 there is no independent labelled reference sample to test the predictions against, and any difference between the three maps is an unknown mixture of real land-cover change and classifier error that cannot be separated without such a reference. The years are also not equally well observed - 16, 48 and 77 cloud-filtered scenes enter the 2018, 2022 and 2024 composites against 80 in 2020 - so the input distribution shifts between years and the 2020 figure is, if anything, an optimistic bound.

# SVMs Applied to Satellite Data

In [ ]:
# Define and train the SVM classifier
class_property = 'Map_remapped'
svm = ee.Classifier.libsvm(kernelType='RBF', gamma=0.5, cost=10).train(
    features=train,
    classProperty=class_property,
    inputProperties=bands
)

# Classify validation and test sets
val_classified = valid.classify(svm)
test_classified = test.classify(svm)

# Evaluate test performance
test_matrix = test_classified.errorMatrix(class_property, 'classification')
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=[f"Actual {l}" for l in label_names],
                       columns=[f"Pred {l}" for l in label_names]))
print("\nClassification Report:")
print(report)

(6) Exercise: train an SVM model using the Landcare NZ 2018 landcover database record for the region. Producing a landcover map of Great Barrier (Aotea) Island for 2018 (based off the Austral summer of 18/19 S2).

Your map should be presented at a publication quality level with all the usual map components (scale, legend, north arrow, data attribution).

You will need to provide performance statistics of the model within your figure.

*   Here you can access the landcover database: https://lris.scinfo.org.nz/layer/104400-lcdb-v50-land-cover-database-version-50-mainland-new-zealand/. You will need to explore for yourself how to extract this data and then upload it to colab, then how to plug it into the SVM algorithim. I have provided some starter code below.

An intial workflow to get the data into the state you need it in to then use it as training data might look like:
- Download the ZIP manually from their browser, having set your area of interest and used the 'Export' tool top right.
- Upload it to Colab.
- Unzip it and load with GeoPandas.
(25 pts)



### Answer (6)**Which database.** The link in the question points at LRIS layer 104400, which is LCDB **v5.0**, whose most recent classification is the 2018/19 summer. The "2024" database is **LCDB v6.0** (released October 2025), which adds a sixth survey period covering the **2023/24** summer in the fields `Class_2023` / `Name_2023`. That is the closest labelled epoch to 2024/25 imagery, so v6.0 is what is used here.**How it was obtained.** Instead of exporting the ZIP by hand from LRIS, the same data is queried from the public Manaaki Whenua feature service published by Eagle Technology / Esri NZ, which needs no login and returns GeoJSON - so this notebook re-runs end to end with no manual download step. (The manual route in the starter code below still works if you prefer it.)

In [ ]:
# ---- Q6 step 1: LCDB v6.0 polygons for Aotea / Great Barrier Island -----------!pip install geopandas rasterio --quietimport requests, json, numpy as np, geopandas as gpdfrom shapely.geometry import Pointfrom shapely.ops import unary_unionLCDB = ('https://services.arcgis.com/XTtANUDT8Va4DLwI/arcgis/rest/services/'        'New_Zealand_Land_Cover/FeatureServer/0/query')ENV = {"xmin": 175.28, "ymin": -36.52, "xmax": 175.62, "ymax": -35.94,       "spatialReference": {"wkid": 4326}}feats, off = [], 0while True:    r = requests.get(LCDB, params=dict(        geometry=json.dumps(ENV), geometryType='esriGeometryEnvelope', inSR=4326,        spatialRel='esriSpatialRelIntersects', outSR=4326, f='geojson', where='1=1',        outFields='OBJECTID,Class_2023,Name_2023,LCDB_UID',        resultOffset=off, resultRecordCount=150, geometryPrecision=7), timeout=180)    got = r.json().get('features', [])    feats += got    if len(got) < 150:        break    off += 150print('LCDB polygons returned:', len(feats))gdf = gpd.GeoDataFrame.from_features(feats, crs=4326)gdf['geometry'] = gdf.geometry.buffer(0)g = gdf[~gdf.geometry.is_empty].to_crs(2193)# the query envelope also clips the northern tip of the Coromandel Peninsula:# keep only the connected components within 5 km of the largest landmasscomps = list(unary_union(g.geometry.values).geoms)main = max(comps, key=lambda c: c.area)aotea = unary_union([c for c in comps if c.distance(main) < 5000])g = g[g.geometry.intersects(aotea)].copy()g['geometry'] = g.geometry.intersection(aotea.buffer(1))g = g[~g.geometry.is_empty]g['area_ha'] = g.geometry.area / 1e4code2name = g.groupby('Class_2023')['Name_2023'].first().to_dict()print(f'{len(g)} polygons, {g.area_ha.sum():.0f} ha, {g.Class_2023.nunique()} classes')

In [ ]:
# ---- Q6 step 2: polygons -> labelled points -> Earth Engine -------------------rng = np.random.default_rng(2)area_by = g.groupby('Class_2023')['area_ha'].sum().sort_values(ascending=False)TOTAL = 6000# area-proportional, but with a floor so the rare classes exist at allalloc = {int(c): int(np.clip(round(TOTAL * a / area_by.sum()), 40, 1200))         for c, a in area_by.items()}pts = []for c, n in alloc.items():    sub = g[g.Class_2023 == c]    w = sub.geometry.area.values; w = w / w.sum()    got = 0    for k in rng.choice(len(sub), size=n * 3, p=w):        if got >= n:            break        poly = sub.geometry.values[k]        minx, miny, maxx, maxy = poly.bounds        for _ in range(60):            p = Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy))            if poly.contains(p):                pts.append((c, p)); got += 1; breakpg = gpd.GeoDataFrame({'lcdb': [p[0] for p in pts]},                      geometry=[p[1] for p in pts], crs=2193).to_crs(4326)print('training points:', len(pg))fc_points = ee.FeatureCollection([    ee.Feature(ee.Geometry.Point([float(p.x), float(p.y)]), {'lcdb': int(c)})    for c, p in zip(pg.lcdb, pg.geometry)])# land mask for clipping the final map (drop the open-water classes)land_gs = gpd.GeoSeries([unary_union(g[~g.Class_2023.isin([20, 21, 22])].geometry.values)                         .simplify(15)], crs=2193).to_crs(4326)land = ee.Geometry(json.loads(land_gs.to_json())['features'][0]['geometry'], None, False)gbi = land.bounds()

In [ ]:
# ---- Q6 step 3: 2024/25 austral-summer composite and the SVM -----------------from sklearn.metrics import precision_recall_fscore_support, accuracy_score, cohen_kappa_scoreALL = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']def mask_all_bands(image):    qa = image.select('QA60')    m = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))    return image.updateMask(m).divide(10000).select(ALL)col25 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(gbi)         .filterDate('2024-12-01', '2025-03-31')         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))print('scenes in the 24/25 austral summer:', col25.size().getInfo())s2_25 = col25.map(mask_all_bands).median().clip(gbi)img25 = (s2_25.addBands(s2_25.normalizedDifference(['B8', 'B4']).rename('NDVI'))              .addBands(s2_25.normalizedDifference(['B3', 'B8']).rename('NDWI'))              .addBands(s2_25.normalizedDifference(['B8', 'B11']).rename('NDMI')))FEATS = ALL + ['NDVI', 'NDWI', 'NDMI']samp = (img25.sampleRegions(collection=fc_points, properties=['lcdb'], scale=10)        .filter(ee.Filter.notNull(FEATS)).randomColumn('random', 2))tr = samp.filter(ee.Filter.lt('random', 0.7))va = samp.filter(ee.Filter.And(ee.Filter.gte('random', 0.7), ee.Filter.lt('random', 0.9)))te = samp.filter(ee.Filter.gte('random', 0.9))n_tr, n_va, n_te = tr.size().getInfo(), va.size().getInfo(), te.size().getInfo()print('train / validation / test:', n_tr, n_va, n_te)# tune on the VALIDATION split only; the test split is touched once, at the endgrid = []for gamma in [5, 50, 200]:    for cost in [10, 100, 1000]:        m = ee.Classifier.libsvm(kernelType='RBF', gamma=gamma, cost=cost).train(            features=tr, classProperty='lcdb', inputProperties=FEATS)        a = va.classify(m).errorMatrix('lcdb', 'classification').accuracy().getInfo()        grid.append((gamma, cost, a))        print(f'  gamma={gamma:<4} cost={cost:<5} validation OA={a:.4f}')best = max(grid, key=lambda t: t[2])print('best:', best)svm = ee.Classifier.libsvm(kernelType='RBF', gamma=best[0], cost=best[1]).train(    features=tr, classProperty='lcdb', inputProperties=FEATS)tested = te.classify(svm)yt = tested.aggregate_array('lcdb').getInfo()yp = tested.aggregate_array('classification').getInfo()oa6  = accuracy_score(yt, yp); kap6 = cohen_kappa_score(yt, yp)mp6, mr6, mf6, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)wp6, wr6, wf6, _ = precision_recall_fscore_support(yt, yp, average='weighted', zero_division=0)print(f'TEST  OA={oa6:.3f}  kappa={kap6:.3f}  macro F1={mf6:.3f}  weighted F1={wf6:.3f}')

In [ ]:
# ---- Q6 step 4: classify, download, and draw the map --------------------------present = sorted(set(yt) | set(yp))P6, R6, F16, SUP6 = precision_recall_fscore_support(yt, yp, labels=present, zero_division=0)classified = img25.select(FEATS).classify(svm).clip(land).unmask(0).toByte()a6, ext6 = fetch(classified, gbi, scale=20, tiles=6)   # SVM inference is memory-hungryarr6 = a6[0]rgb_img = s2_25.select(['B4', 'B3', 'B2']).divide(0.3).clamp(0, 1).multiply(255).unmask(0).toByte()a_rgb, rext6 = fetch(rgb_img, gbi, scale=20, tiles=3)rgb6 = np.dstack([a_rgb[0], a_rgb[1], a_rgb[2]])LCDB_COL = {0: '#ffffff', 1: '#c8102e', 2: '#ff9d9d', 5: '#5a5a5a', 6: '#8b5a2b',            10: '#ffeeba', 12: '#d2a679', 16: '#bdbdbd', 20: '#1f4e9c', 21: '#4a90d9',            22: '#8ecae6', 33: '#e08fe0', 40: '#ffff8c', 41: '#e0d24a', 45: '#00b3a4',            46: '#6fc9a8', 47: '#9acd32', 51: '#d9a300', 52: '#a8d96a', 54: '#4d9221',            64: '#c9a227', 69: '#12591f', 70: '#006b5b', 71: '#7b52a0'}codes = [c for c in sorted(set(np.unique(arr6).tolist()) | set(present) | {0})         if c in LCDB_COL]lut = np.zeros(max(codes) + 1, dtype=int)for i, c in enumerate(codes):    lut[c] = icmap6 = ListedColormap([LCDB_COL[c] for c in codes])norm6 = BoundaryNorm(np.arange(-0.5, len(codes) + 0.5, 1), len(codes))ASP = 1 / np.cos(np.radians(36.25))fig = plt.figure(figsize=(6.268, 7.05), dpi=300)gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1.53], height_ratios=[3.62, 2.42],                      wspace=0.05, hspace=0.10, left=0.012, right=0.988,                      top=0.938, bottom=0.070)axm = fig.add_subplot(gs[0, 0])axm.imshow(np.ma.masked_equal(lut[np.clip(arr6, 0, max(codes))], 0), extent=ext6,           cmap=cmap6, norm=norm6, interpolation='nearest', aspect=ASP)axm.set_facecolor('#dbeaf5'); axm.set_xticks([]); axm.set_yticks([])axm.set_title('(a)  SVM land cover 2024/25', fontsize=7.4, fontweight='bold',              pad=2.5, loc='left')scalebar(axm, ext6, km=10, y=0.030, x=0.07, fs=5.4)axm.annotate('', xy=(0.885, 0.963), xytext=(0.885, 0.90), xycoords='axes fraction',             arrowprops=dict(arrowstyle='-|>', lw=1.1, color='k', mutation_scale=9))axm.text(0.885, 0.968, 'N', transform=axm.transAxes, ha='center', va='bottom',         fontsize=7, fontweight='bold')axr = fig.add_subplot(gs[0, 1])axr.imshow(rgb6, extent=rext6, interpolation='nearest', aspect=ASP)axr.set_xlim(ext6[0], ext6[1]); axr.set_ylim(ext6[2], ext6[3])axr.set_xticks([]); axr.set_yticks([])axr.set_title('(b)  Sentinel-2, same composite', fontsize=7.4, fontweight='bold',              pad=2.5, loc='left')axl = fig.add_subplot(gs[0, 2]); axl.axis('off')axl.set_title('(c)  LCDB v6.0 class (2023/24 survey)', fontsize=7.4, fontweight='bold',              pad=2.5, loc='left')show = [c for c in codes if c != 0]step = 0.965 / len(show)for i, c in enumerate(show):    y = 0.985 - i*step    axl.add_patch(Rectangle((0.010, y-0.0165), 0.055, 0.021, transform=axl.transAxes,                            fc=LCDB_COL[c], ec='0.35', lw=0.35, clip_on=False))    axl.text(0.080, y-0.006, f'{c}  {code2name.get(c, c)}', transform=axl.transAxes,             fontsize=5.6, va='center')axt = fig.add_subplot(gs[1, 0:2]); axt.axis('off')axt.set_title(f'(d)  SVM performance, held-out 10 % test split (n = {len(yt)})',              fontsize=7.4, fontweight='bold', pad=2.5, loc='left')tb = axt.table(cellText=[['Overall accuracy', f'{oa6:.3f}'],                         ['Cohen kappa', f'{kap6:.3f}'],                         ['Macro avg  precision / recall / F1',                          f'{mp6:.2f} / {mr6:.2f} / {mf6:.2f}'],                         ['Weighted avg  precision / recall / F1',                          f'{wp6:.2f} / {wr6:.2f} / {wf6:.2f}'],                         ['Kernel (tuned on the validation split)',                          f'RBF, gamma {best[0]}, C {best[1]}'],                         ['Features', '10 S2 bands + NDVI, NDWI, NDMI'],                         ['Train / validation / test points', f'{n_tr} / {n_va} / {n_te}']],               colWidths=[0.60, 0.40], cellLoc='left', bbox=[0.010, 0.365, 0.980, 0.615])tb.auto_set_font_size(False); tb.set_fontsize(5.9)for (r, c), cell in tb.get_celld().items():    cell.set_linewidth(0.3); cell.set_edgecolor('0.78')    if c == 0: cell.set_facecolor('#f2f2f2')axf = fig.add_subplot(gs[1, 2])o = np.argsort(-F16)axf.barh(np.arange(len(present)), F16[o],         color=[LCDB_COL[present[k]] for k in o], edgecolor='0.35', lw=0.3)axf.set_yticks(np.arange(len(present)))axf.set_yticklabels([f'{str(code2name.get(present[k], present[k]))[:26]} ({SUP6[k]})'                     for k in o], fontsize=4.5)axf.invert_yaxis(); axf.set_xlim(0, 1)axf.set_xlabel('per-class F1 on the test split (n)', fontsize=5.6)axf.tick_params(labelsize=5)axf.axvline(oa6, color='0.2', ls='--', lw=0.7)axf.set_title('(e)  Per-class F1', fontsize=7.4, fontweight='bold', pad=2.5, loc='left')fig.text(0.012, 0.006,         'Training labels: Land Cover Database v6.0, 2023/24 summer survey (Manaaki Whenua '         '- Landcare Research, CC BY 4.0, served by Eagle Technology / Esri NZ).\n'         'Imagery: Copernicus Sentinel-2 SR Harmonized, 1 Dec 2024 - 31 Mar 2025, scene '         'cloud < 10 %, QA60 mask, median composite, 20 m. Classifier: ee.Classifier.libsvm.\n'         'Geographic coordinates (WGS 84); map clipped to the LCDB land extent. '         'Processed in Google Earth Engine.', fontsize=4.6, color='0.3', va='bottom')fig.suptitle('Land cover of Aotea / Great Barrier Island, austral summer 2024/25, '             'from an SVM trained on LCDB v6.0',             fontsize=7.9, fontweight='bold', x=0.012, ha='left', y=0.988)fig.savefig('fig_q6.png', dpi=300, facecolor='white')plt.show()

**Figure 3.** Land cover of Aotea / Great Barrier Island for the austral summer of 2024/25, from a support vector machine (RBF kernel) trained on the Manaaki Whenua Land Cover Database v6.0, 2023/24 survey.**Result.** Overall accuracy **0.535**, Cohen kappa **0.417** on 456 held-out points (macro F1 0.45, weighted F1 0.51). Repeating the whole procedure with only the four lab bands plus NDVI, tuned separately, gives 0.520 / 0.388 with a macro F1 of 0.33 - the extra bands buy little overall accuracy but a great deal on the small classes.**How to read that number.** 0.54 looks poor beside the 0.90 the Wellington random forest returned, but the two are not comparable: 22 classes rather than six; no single dominant class (in Wellington, predicting water everywhere would already have scored 0.51); and the LCDB legend deliberately splits categories that are close to spectrally identical in a single summer composite - Manuka and/or Kanuka, Broadleaved Indigenous Hardwoods and Indigenous Forest alone supply 71 % of the test points and account for most of the confusion. A guess drawn from the class frequencies would score about 0.19, so kappa = 0.42 is real signal, and the spectrally distinct classes behave well (lake, estuarine open water, mangrove, herbaceous freshwater vegetation and surface mine all reach F1 between 0.67 and 1.00).Two limits are structural rather than a matter of tuning. LCDB is mapped to a 1 ha minimum unit and was delineated from imagery a year before this composite, so points near polygon edges carry the wrong label and genuine change between 2023/24 and 2024/25 counts as error. And LCDB encodes *land use* as well as land cover - "Forest - Harvested", "Urban Parkland/Open Space" - which no purely spectral classifier can recover; both score F1 = 0.00 here.

In [ ]:
# Code to get you started
import zipfile
import geopandas as gpd

# Upload the ZIP manually using the Colab UI
from google.colab import files
uploaded = files.upload()  # <- Expects a ZIP

# Unzip
with zipfile.ZipFile("LCDB_v5.zip", 'r') as zip_ref: #<- Check file names
    zip_ref.extractall("lcdb")

# Read shapefile
gdf = gpd.read_file("lcdb/LCDB_v5.shp") #<- Check file names
print(gdf.head())
